# ECG Classification – Eksperimen Skema Preprocessing

Notebook ini melatih model klasifikasi ECG dengan **3 skema preprocessing** dan **2 task klasifikasi**, sehingga total terdapat **6 kombinasi eksperimen**.

| Skema | Input | Task |
|---|---|---|
| Skema 1 | Gambar clean (1 gambar/sampel) | 2-class & 4-class |
| Skema 2 | 12 short lead + 1 long lead | 2-class & 4-class |
| Skema 3 | 6 lead tungkai + 6 lead precordial + long lead | 2-class & 4-class |

Semua eksperimen di-tracking menggunakan **MLflow via DagsHub**.

## 1. Import & Konfigurasi

In [ ]:
import torch
from utils.config import Config
from utils.modeling import (
    init_dagshub,
    CLASS_NAMES_2, CLASS_NAMES_4,
    SCHEME_NAMES,
)

In [ ]:
# ── Hyperparameter ──────────────────────────────────────────────
NUM_EPOCHS    = 20
BATCH_SIZE    = 16
LEARNING_RATE = 1e-4
IMAGE_SIZE    = 224
VAL_SPLIT     = 0.15
TEST_SPLIT    = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Inisialisasi DagsHub & MLflow

In [ ]:
init_dagshub(
    repo_owner=Config.dagshub_config["repo_owner"],
    repo_name=Config.dagshub_config["repo_name"],
)

## 3. Skema 1 – Gambar Clean (1 Gambar per Sampel)

Input: satu gambar ECG bersih per pasien dari `data/preprocessed/clean_ecg_signal/`.

### 3.1 Skema 1 – Task 4-Class (Normal / Abnormal / MI / History MI)

In [ ]:
from utils.modeling import (
    _get_image_files_scheme1,
    ECGDatasetScheme1,
    build_dataloaders,
    build_model,
    train_model,
    evaluate_and_log,
)

records_s1 = _get_image_files_scheme1(Config.data_root)

dataset_s1_4c = ECGDatasetScheme1(records_s1, task="4class", image_size=IMAGE_SIZE)
train_s1_4c, val_s1_4c, test_s1_4c = build_dataloaders(
    dataset_s1_4c, seed=Config.seed, batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT, test_split=TEST_SPLIT
)

model_s1_4c = build_model(num_classes=4, in_channels=3).to(DEVICE)

model_s1_4c, history_s1_4c = train_model(
    model_s1_4c, train_s1_4c, val_s1_4c,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
    device=DEVICE, run_name="scheme1_4class",
    scheme="scheme1", task="4class"
)

In [ ]:
result_s1_4c = evaluate_and_log(
    model_s1_4c, test_s1_4c, DEVICE,
    class_names=CLASS_NAMES_4,
    scheme="scheme1", task="4class"
)

### 3.2 Skema 1 – Task 2-Class (Sehat / Sakit)

In [ ]:
dataset_s1_2c = ECGDatasetScheme1(records_s1, task="2class", image_size=IMAGE_SIZE)
train_s1_2c, val_s1_2c, test_s1_2c = build_dataloaders(
    dataset_s1_2c, seed=Config.seed, batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT, test_split=TEST_SPLIT
)

model_s1_2c = build_model(num_classes=2, in_channels=3).to(DEVICE)

model_s1_2c, history_s1_2c = train_model(
    model_s1_2c, train_s1_2c, val_s1_2c,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
    device=DEVICE, run_name="scheme1_2class",
    scheme="scheme1", task="2class"
)

In [ ]:
result_s1_2c = evaluate_and_log(
    model_s1_2c, test_s1_2c, DEVICE,
    class_names=CLASS_NAMES_2,
    scheme="scheme1", task="2class"
)

## 4. Skema 2 – 12 Short Lead + 1 Long Lead

Input: 13 gambar per pasien dari `data/preprocessed/cropped_leads/`, di-stack menjadi satu tensor (39 channel).

### 4.1 Skema 2 – Task 4-Class

In [ ]:
from utils.modeling import _get_image_files_scheme2, ECGDatasetScheme2

records_s2 = _get_image_files_scheme2(Config.data_root)

dataset_s2_4c = ECGDatasetScheme2(records_s2, task="4class", image_size=IMAGE_SIZE)
train_s2_4c, val_s2_4c, test_s2_4c = build_dataloaders(
    dataset_s2_4c, seed=Config.seed, batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT, test_split=TEST_SPLIT
)

# 13 leads x 3 channels = 39 input channels
model_s2_4c = build_model(num_classes=4, in_channels=39).to(DEVICE)

model_s2_4c, history_s2_4c = train_model(
    model_s2_4c, train_s2_4c, val_s2_4c,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
    device=DEVICE, run_name="scheme2_4class",
    scheme="scheme2", task="4class"
)

In [ ]:
result_s2_4c = evaluate_and_log(
    model_s2_4c, test_s2_4c, DEVICE,
    class_names=CLASS_NAMES_4,
    scheme="scheme2", task="4class"
)

### 4.2 Skema 2 – Task 2-Class

In [ ]:
dataset_s2_2c = ECGDatasetScheme2(records_s2, task="2class", image_size=IMAGE_SIZE)
train_s2_2c, val_s2_2c, test_s2_2c = build_dataloaders(
    dataset_s2_2c, seed=Config.seed, batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT, test_split=TEST_SPLIT
)

model_s2_2c = build_model(num_classes=2, in_channels=39).to(DEVICE)

model_s2_2c, history_s2_2c = train_model(
    model_s2_2c, train_s2_2c, val_s2_2c,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
    device=DEVICE, run_name="scheme2_2class",
    scheme="scheme2", task="2class"
)

In [ ]:
result_s2_2c = evaluate_and_log(
    model_s2_2c, test_s2_2c, DEVICE,
    class_names=CLASS_NAMES_2,
    scheme="scheme2", task="2class"
)

## 5. Skema 3 – 6 Lead Tungkai + 6 Lead Precordial + Long Lead

Input: sumber data sama dengan Skema 2, namun urutan lead dikelompokkan secara anatomis:
- **Lead Tungkai (Limb):** I, II, III, aVR, aVL, aVF
- **Lead Precordial (Chest):** V1, V2, V3, V4, V5, V6
- **Long Lead:** 1 lead rhythm strip

### 5.1 Skema 3 – Task 4-Class

In [ ]:
from utils.modeling import _get_image_files_scheme3, ECGDatasetScheme3

records_s3 = _get_image_files_scheme3(Config.data_root)

dataset_s3_4c = ECGDatasetScheme3(records_s3, task="4class", image_size=IMAGE_SIZE)
train_s3_4c, val_s3_4c, test_s3_4c = build_dataloaders(
    dataset_s3_4c, seed=Config.seed, batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT, test_split=TEST_SPLIT
)

model_s3_4c = build_model(num_classes=4, in_channels=39).to(DEVICE)

model_s3_4c, history_s3_4c = train_model(
    model_s3_4c, train_s3_4c, val_s3_4c,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
    device=DEVICE, run_name="scheme3_4class",
    scheme="scheme3", task="4class"
)

In [ ]:
result_s3_4c = evaluate_and_log(
    model_s3_4c, test_s3_4c, DEVICE,
    class_names=CLASS_NAMES_4,
    scheme="scheme3", task="4class"
)

### 5.2 Skema 3 – Task 2-Class

In [ ]:
dataset_s3_2c = ECGDatasetScheme3(records_s3, task="2class", image_size=IMAGE_SIZE)
train_s3_2c, val_s3_2c, test_s3_2c = build_dataloaders(
    dataset_s3_2c, seed=Config.seed, batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT, test_split=TEST_SPLIT
)

model_s3_2c = build_model(num_classes=2, in_channels=39).to(DEVICE)

model_s3_2c, history_s3_2c = train_model(
    model_s3_2c, train_s3_2c, val_s3_2c,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
    device=DEVICE, run_name="scheme3_2class",
    scheme="scheme3", task="2class"
)

In [ ]:
result_s3_2c = evaluate_and_log(
    model_s3_2c, test_s3_2c, DEVICE,
    class_names=CLASS_NAMES_2,
    scheme="scheme3", task="2class"
)

## 6. Visualisasi Perbandingan Hasil

Bagian ini membandingkan performa semua skema dan task secara visual.

### 6.1 Training History – Task 4-Class

In [ ]:
from utils.modeling import plot_training_history

plot_training_history(
    histories={
        "Skema 1 – 4 Class": history_s1_4c,
        "Skema 2 – 4 Class": history_s2_4c,
        "Skema 3 – 4 Class": history_s3_4c,
    },
    title_prefix="4-Class | "
)

### 6.2 Training History – Task 2-Class

In [ ]:
plot_training_history(
    histories={
        "Skema 1 – 2 Class": history_s1_2c,
        "Skema 2 – 2 Class": history_s2_2c,
        "Skema 3 – 2 Class": history_s3_2c,
    },
    title_prefix="2-Class | "
)

### 6.3 Confusion Matrix – Task 4-Class

In [ ]:
from utils.modeling import plot_confusion_matrices

plot_confusion_matrices(
    results={
        "Skema 1 – 4 Class": result_s1_4c,
        "Skema 2 – 4 Class": result_s2_4c,
        "Skema 3 – 4 Class": result_s3_4c,
    },
    class_names=CLASS_NAMES_4
)

### 6.4 Confusion Matrix – Task 2-Class

In [ ]:
plot_confusion_matrices(
    results={
        "Skema 1 – 2 Class": result_s1_2c,
        "Skema 2 – 2 Class": result_s2_2c,
        "Skema 3 – 2 Class": result_s3_2c,
    },
    class_names=CLASS_NAMES_2
)

### 6.5 Perbandingan Akhir – Accuracy & Macro F1 Semua Skema

In [ ]:
from utils.modeling import plot_scheme_comparison

all_results = {
    "Skema1\n4-Class": result_s1_4c,
    "Skema1\n2-Class": result_s1_2c,
    "Skema2\n4-Class": result_s2_4c,
    "Skema2\n2-Class": result_s2_2c,
    "Skema3\n4-Class": result_s3_4c,
    "Skema3\n2-Class": result_s3_2c,
}

plot_scheme_comparison(all_results)

### 6.6 Sample Predictions – Skema Terbaik

Visualisasi prediksi model pada sampel acak dari dataset test. Ganti variabel `best_model` dan `best_dataset` sesuai skema dengan performa terbaik dari grafik di atas.

In [ ]:
from utils.modeling import plot_sample_predictions

# Ganti model dan dataset sesuai hasil terbaik yang diperoleh
best_model   = model_s1_4c
best_dataset = dataset_s1_4c
best_classes = CLASS_NAMES_4

plot_sample_predictions(
    model=best_model,
    dataset=best_dataset,
    class_names=best_classes,
    device=DEVICE,
    num_samples=8
)